In [1]:
print('hii')

hii


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00


# Step 1: Download & Filter VisDrone

In [3]:
import os
import shutil
from pathlib import Path
from PIL import Image
from ultralytics.utils.downloads import download
from ultralytics.utils import ASSETS_URL, TQDM

def visdrone_to_single_class(dir, split, source_name=None):
    """Convert VisDrone to single-class YOLO format (only pedestrian & people mapped to 0)."""
    source_dir = dir / (source_name or f"VisDrone2019-DET-{split}")
    images_dir = dir / "images" / split
    labels_dir = dir / "labels" / split
    labels_dir.mkdir(parents=True, exist_ok=True)

    # Move images
    if (source_images_dir := source_dir / "images").exists():
        images_dir.mkdir(parents=True, exist_ok=True)
        for img in source_images_dir.glob("*.jpg"):
            img.rename(images_dir / img.name)

    # Process annotations
    for f in TQDM((source_dir / "annotations").glob("*.txt"), desc=f"Filtering VisDrone {split}"):
        img_path = images_dir / f.with_suffix(".jpg").name
        if not img_path.exists():
            continue
            
        img_size = Image.open(img_path).size
        dw, dh = 1.0 / img_size[0], 1.0 / img_size[1]
        lines = []

        with open(f, encoding="utf-8") as file:
            for row in [x.split(",") for x in file.read().strip().splitlines()]:
                if row[4] != "0":  # Skip ignored regions
                    cls = int(row[5]) - 1 # Original class index
                    
                    # 0: pedestrian, 1: people -> map both to 0 (person)
                    if cls in [0, 1]:
                        x, y, w, h = map(int, row[:4])
                        x_center, y_center = (x + w / 2) * dw, (y + h / 2) * dh
                        w_norm, h_norm = w * dw, h * dh
                        lines.append(f"0 {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

        # Only write label file if it contains target objects
        if lines:
            (labels_dir / f.name).write_text("".join(lines), encoding="utf-8")

# Setup paths and download
visdrone_dir = Path("./VisDrone")
urls = [
    f"{ASSETS_URL}/VisDrone2019-DET-train.zip",
    f"{ASSETS_URL}/VisDrone2019-DET-val.zip",
]
download(urls, dir=visdrone_dir, threads=4)

# Process splits
splits = {"VisDrone2019-DET-train": "train", "VisDrone2019-DET-val": "val"}
for folder, split in splits.items():
    visdrone_to_single_class(visdrone_dir, split, folder)
    shutil.rmtree(visdrone_dir / folder) # clean source folder
print("VisDrone filtering complete!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Unzipping VisDrone/VisDrone2019-DET-val.zip to /kaggle/working/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.5Kfiles/s 0.8s
Unzipping VisDrone/VisDrone2019-DET-train.zip to /kaggle/working/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.7Kfiles/s 7.5s
Filtering VisDrone train: ━━━━━━━━━━━━ 6471 3.3Kit/s 2.1s
Filtering VisDrone val: ━━━━━━━━━━━━ 548 1.7Kit/s 0.2s
VisDrone filtering complete!


# Step 2: Download the Roboflow MOT Dataset

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="PdufbKJSUUS5xZED1Vrw")
project = rf.workspace("rahul-kishore-gorai").project("mot-ma3bf-5b96r")
version = project.version(1)
mot_dataset = version.download("yolo26")

# Find where Roboflow downloaded it (usually a folder named 'mot-1' or similar in current directory)
mot_dir = Path(mot_dataset.location).resolve()
visdrone_dir = Path("./VisDrone").resolve()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 96.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, 


Extracting Dataset Version Zip to MOT-1 in yolo26:: 100%|██████████| 18346/18346 [00:04<00:00, 4235.29it/s]


# Step 3: Create the Combined Data Configuration File

In [5]:
import yaml

combined_yaml = {
    'path': '', # Leave blank if using absolute paths below
    'train': [
        str(visdrone_dir / 'images/train'),
        str(mot_dir / 'train/images')
    ],
    'val': [
        str(visdrone_dir / 'images/val'),
        str(mot_dir / 'valid/images') # Verify if Roboflow named it 'valid' or 'val'
    ],
    'names': {
        0: 'person'
    }
}

with open('combined_person.yaml', 'w') as f:
    yaml.dump(combined_yaml, f, default_flow_style=False)

In [6]:
import os
import cv2
import random
import torch
import numpy as np
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from ultralytics import YOLO
from ultralytics.data.augment import Albumentations
from ultralytics.utils import LOGGER, colorstr

class AddScanLines(ImageOnlyTransform):
    """
    Adds simulated scan lines (LED strips) to an image, mimicking the effect of photographing a screen.
    """
    def __init__(self, line_density=10, line_thickness_range=(1, 5), opacity_range=(0.1, 0.3), always_apply=False, p=1.0):
        super(AddScanLines, self).__init__(p=p)
        self.line_density = line_density
        self.line_thickness_range = line_thickness_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        height, width, _ = image.shape
        image_with_lines = image.copy()
        num_lines = random.randint(5, self.line_density)

        for _ in range(num_lines):
            y_start = random.randint(0, height - 1)
            y_end = y_start + random.randint(1, 5)  
            
            line_thickness = random.randint(*self.line_thickness_range)
            opacity = random.uniform(*self.opacity_range)
            line_color = (random.randint(50, 150), random.randint(50, 150), random.randint(50, 150))
            
            overlay = image_with_lines.copy()
            cv2.line(overlay, (0, y_start), (width, y_end), line_color, line_thickness)
            cv2.addWeighted(overlay, opacity, image_with_lines, 1 - opacity, 0, image_with_lines)

        return image_with_lines
    
    def get_transform_init_args_names(self):
        return ("line_density", "line_thickness_range", "opacity_range")
    
class AddMoirePattern(ImageOnlyTransform):
    """
    Adds synthetic moiré patterns to the image by overlaying a rotated and shifted grid pattern.
    """
    def __init__(self, grid_density=20, rotation_range=(-15, 15), opacity_range=(0.05, 0.2), always_apply=False, p=1.0):
        super(AddMoirePattern, self).__init__(p)
        self.grid_density = grid_density
        self.rotation_range = rotation_range
        self.opacity_range = opacity_range

    def apply(self, image, **params):
        image = image.copy() 
        height, width, _ = image.shape
        
        grid = np.zeros((height, width), dtype=np.uint8)
        step_size = max(1, min(width, height) // self.grid_density)
        
        for i in range(0, width, step_size):
            cv2.line(grid, (i, 0), (i, height), 255, 1)
        for i in range(0, height, step_size):
            cv2.line(grid, (0, i), (width, i), 255, 1)
        
        rotation_angle = random.uniform(*self.rotation_range)
        rotation_matrix = cv2.getRotationMatrix2D((width // 2, height // 2), rotation_angle, 1)
        rotated_grid = cv2.warpAffine(grid, rotation_matrix, (width, height))
        
        opacity = random.uniform(*self.opacity_range)
        grid_rgb = cv2.cvtColor(rotated_grid, cv2.COLOR_GRAY2BGR)
        
        image_with_moire = cv2.addWeighted(image, 1 - opacity, grid_rgb, opacity, 0)
        return image_with_moire   
    
    def get_transform_init_args_names(self):
        return ("grid_density", "rotation_range", "opacity_range")


# UPDATED SIGNATURE: Added transforms=None and **kwargs to absorb Ultralytics updates
def custom_albumentations_init(self, p=1.0, transforms=None, **kwargs):
    """
    Initialize the transform object for YOLO bbox formatted params.
    This replaces Ultralytics default Albumentations init.
    """
    self.p = p
    self.transform = None
    self.contains_spatial = True
    prefix = colorstr("albumentations: ")
    try:
        print("Attempting to compose custom Albumentations pipeline...")
        
        T = [
            # Pixel-level transforms 
            A.Blur(blur_limit=3, p=0.01),
            A.ChannelDropout(p=0.3),
            A.ChannelShuffle(p=0.3),
            A.ChromaticAberration(p=0.2),
            A.CLAHE(p=0.1),
            A.ColorJitter(p=0.2),
            A.Defocus(p=0.0001),
            A.Emboss(p=0.01),
            A.FancyPCA(alpha=0.1, p=0.2),
            A.GaussianBlur(blur_limit=(1, 3), p=0.01),
            A.GaussNoise(var_limit=(10, 20), p=0.01),
            A.GlassBlur(sigma=0.6, max_delta=3, iterations=1, p=0.001),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=50, val_shift_limit=10, p=0.2),
            A.ISONoise(p=0.3),
            A.InvertImg(p=0.01),
            A.MedianBlur(blur_limit=3, p=0.01),
            A.MotionBlur(blur_limit=3, p=0.05),
            A.MultiplicativeNoise(p=0.1),
            A.PlanckianJitter(p=0.2),
            A.RandomBrightnessContrast(p=0.1),
            A.RandomFog(p=0.1),
            A.RandomGamma(gamma_limit=(80, 120), p=0.1),
            A.RandomToneCurve(p=0.2),
            A.RingingOvershoot(p=0.005),
            A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.2),
            A.ToSepia(p=0.1),
            A.Sharpen(p=0.1),
            A.Spatter(p=0.005),
            A.Superpixels(p=0.001),
            A.ToGray(p=0.3),
            A.UnsharpMask(p=0.05),
            
            # Spatial transforms
            A.HorizontalFlip(p=0.3),              
            A.VerticalFlip(p=0.3),                    
            A.RandomCrop(width=300, height=300, p=0.1), 
            A.Rotate(limit=90, p=0.1),                
            
            # Custom transforms
            AddScanLines(line_density=105, line_thickness_range=(1, 4), opacity_range=(0.1, 0.3), p=0.1),
            AddMoirePattern(grid_density=30, rotation_range=(-10, 10), opacity_range=(0.05, 0.15), p=0.2),
            
            # Final Resize
            A.Resize(height=640, width=640, p=1.0),
        ]
        
        self.transform = A.Compose(
            T, 
            bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3)
        )
        
        print("Transformations successfully composed.") 
        LOGGER.info(prefix + ", ".join(f"{x}".replace("always_apply=False, ", "") for x in T if getattr(x, 'p', 1.0)))
        
    except ImportError:  
        print("Albumentations not installed. Skipping")
        pass
    except Exception as e:
        LOGGER.info(f"{prefix}Error: {e}")

# 1. Apply the monkey patch to override Ultralytics Albumentations
Albumentations.__init__ = custom_albumentations_init


if __name__ == '__main__':
    # 2. Load model
    model = YOLO('yolov8s.pt')

    # 3. Train STAGE 1 (Frozen Warm-Up)
    print("Starting Stage 1: Frozen Backbone Warm-up...")
    results = model.train(
        data='combined_person.yaml', 
        freeze=list(range(12)),  # Fully freeze backbone (0-11) for safe head alignment
        epochs=15,               # Short warm-up run
        batch=48, 
        imgsz=640, 
        workers=4, 
        device='0,1',
        project='runs/train',
        name='stage1_frozen'
    )

Starting Stage 1: Frozen Backbone Warm-up...
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=combined_person.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, 

In [7]:
import os
from pathlib import Path
from ultralytics import YOLO

if __name__ == '__main__':
    # Locate the best weights from the Stage 1 training run
    stage1_weights = Path('runs/detect/runs/train/stage1_frozen/weights/best.pt').resolve()
    
    if not stage1_weights.exists():
        raise FileNotFoundError("Stage 1 weights not found. Please ensure Cell 1 completed successfully.")

    # Load the model that now has a stable 1-class head
    model_stage2 = YOLO(str(stage1_weights))

    # Train STAGE 2 (Unfrozen)
    print("\nStarting Stage 2: Unfrozen Fine-tuning...")
    results_stage2 = model_stage2.train(
        data='combined_person.yaml', 
        # No 'freeze' argument so the entire network updates
        epochs=150,              # Ample time for backbone to learn drone/MOT perspectives
        batch=16, 
        imgsz=640, 
        workers=4, 
        device='0,1',
        lr0=0.001,               # Lower learning rate protects foundational COCO knowledge
        patience=25,             # Safe buffer for early stopping if it plateaus early
        project='runs/train',
        name='stage2_unfrozen'
    )
    
    print("Training complete! Final optimized weights saved to: runs/train/stage2_unfrozen/weights/best.pt")


Starting Stage 2: Unfrozen Fine-tuning...
Ultralytics 8.4.53 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=combined_person.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/detect/runs/train/stage1_

In [8]:
import shutil
from pathlib import Path

# Define paths
source_dir = Path('runs/detect').resolve()
output_zip_name = 'yolov8_two_stage_results'  # This will create 'yolov8_two_stage_results.zip'

if source_dir.exists():
    print(f"Zipping training results from {source_dir}...")
    
    # Pack the folder into a zip archive
    # shutil.make_archive automatically appends '.zip' to the output name
    archive_path = shutil.make_archive(output_zip_name, 'zip', source_dir)
    
    print(f"Successfully created zip archive at: {archive_path}")
    print(f"Size: {Path(archive_path).stat().st_size / (1024 * 1024):.2f} MB")
else:
    print(f"Error: The directory '{source_dir}' does not exist. Did you change your training 'project' path?")

Zipping training results from /kaggle/working/runs/detect...
Successfully created zip archive at: /kaggle/working/yolov8_two_stage_results.zip
Size: 91.29 MB
